# HAA Leader Sector System v2.1 — Detailed Order Review Runner

실제 현금·보유종목 기준 승인 주문 후보를 사람이 검토할 수 있는 상세 표로 표시합니다. 실제 `run_daily()`와 파일 저장·주문 실행은 호출하지 않습니다.

필수 준비: Colab Secrets `KRX_ID`, `KRX_PW`, `HAA_CURRENT_CASH`; Drive 파일 `/MyDrive/HAA_Leader_System/portfolio/current_holdings.csv`.
현재 보유종목이 없으면 CSV는 헤더만 유지합니다. 가격은 주문 가격이 아니라 계산 시점의 참고가격입니다.


In [ ]:
REPOSITORY = "hyunsungkim73/HAA_Leader_Sector_System"
SOURCE_REF = "codex/haa-v2.1-order-review-preview"  # 검증 후 main으로 전환
SOURCE_NOTEBOOK = "HAA_Leader_Sector_System_v2_1.ipynb"
HOLDINGS_CSV = "/content/drive/MyDrive/HAA_Leader_System/portfolio/current_holdings.csv"


In [ ]:
import os
import time
from google.colab import userdata

def load_colab_secret(secret_name, max_attempts=4):
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            secret_value = userdata.get(secret_name)
            if not str(secret_value).strip():
                raise RuntimeError(f"Colab Secret {secret_name!r} is empty.")
            return str(secret_value).strip()
        except Exception as e:
            last_error = e
            error_text = str(e)
            transient = (
                "503" in error_text
                or "Could not fetch resource" in error_text
                or "FetchError" in error_text
            )
            if transient and attempt < max_attempts:
                wait_seconds = 2 ** attempt
                print(
                    f"COLAB_SECRET_RETRY name={secret_name} "
                    f"attempt={attempt}/{max_attempts} wait={wait_seconds}s"
                )
                time.sleep(wait_seconds)
                continue
            if transient:
                raise RuntimeError(
                    f"Colab Secrets service is temporarily unavailable while "
                    f"reading {secret_name!r} (HTTP 503). Restart the Colab "
                    "session or retry after a few minutes; do not recreate or "
                    "expose the Secret value."
                ) from e
            raise RuntimeError(
                f"Colab Secret {secret_name!r} does not exist or notebook "
                "access is disabled. Check the key icon in the left sidebar."
            ) from e
    raise RuntimeError(
        f"Unable to read Colab Secret {secret_name!r}."
    ) from last_error

for secret_name in ("KRX_ID", "KRX_PW", "HAA_CURRENT_CASH"):
    os.environ[secret_name] = load_colab_secret(secret_name)

if "@" in os.environ["KRX_ID"]:
    raise RuntimeError(
        "KRX_ID must be the KRX data-portal member ID, not an email address."
    )

cash_text = os.environ["HAA_CURRENT_CASH"].replace(",", "")
try:
    current_cash = float(cash_text)
except ValueError as e:
    raise RuntimeError("HAA_CURRENT_CASH must be numeric.") from e
if current_cash < 0:
    raise RuntimeError("HAA_CURRENT_CASH must be zero or positive.")
os.environ["HAA_CURRENT_CASH"] = str(current_cash)
print("Colab Secrets loaded (values are not displayed).")


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
holdings_path = Path(HOLDINGS_CSV)
if not holdings_path.is_file():
    raise FileNotFoundError(
        f"Holdings CSV not found: {holdings_path}. "
        "Copy current_holdings_template.csv to this location and fill it first."
    )

header = holdings_path.read_text(encoding="utf-8-sig").splitlines()[0]
required_headers = {
    "ticker", "name", "sector", "asset_type", "quantity", "avg_price"
}
actual_headers = {item.strip() for item in header.split(",")}
missing_headers = sorted(required_headers - actual_headers)
if missing_headers:
    raise RuntimeError(
        "Holdings CSV is missing columns: " + ", ".join(missing_headers)
    )

os.environ["HAA_HOLDINGS_CSV"] = str(holdings_path)
print(f"HOLDINGS_INPUT_OK path={holdings_path}")


In [ ]:
import base64
import pathlib
import requests

api_url = (
    f"https://api.github.com/repos/{REPOSITORY}/contents/{SOURCE_NOTEBOOK}"
)
response = requests.get(
    api_url,
    params={"ref": SOURCE_REF},
    headers={
        "Accept": "application/vnd.github+json",
        "Cache-Control": "no-cache",
    },
    timeout=30,
)
response.raise_for_status()
payload = response.json()

source_path = pathlib.Path("/content") / SOURCE_NOTEBOOK
source_path.write_bytes(base64.b64decode(payload["content"]))
SOURCE_COMMIT_BLOB_SHA = payload["sha"]
print(f"SYNC_OK ref={SOURCE_REF} blob_sha={SOURCE_COMMIT_BLOB_SHA}")
print(f"Downloaded: {source_path}")


In [ ]:
import json
import subprocess
import sys
import time

executed_path = pathlib.Path(
    "/content/HAA_Leader_Sector_System_v2_1_order_review_executed.ipynb"
)
run_env = os.environ.copy()
run_env["HAA_EXECUTION_CHECK_ONLY"] = "1"
run_env["HAA_AUTO_RUN_KRX_PREFLIGHT"] = "1"
run_env["HAA_DAILY_PREVIEW"] = "0"
run_env["HAA_ACTUAL_HOLDINGS_PREVIEW"] = "1"
run_env["HAA_ORDER_REVIEW_DETAIL"] = "1"

command = [
    sys.executable, "-m", "jupyter", "nbconvert",
    "--to", "notebook", "--execute",
    "--ExecutePreprocessor.timeout=3600",
    "--output", str(executed_path), str(source_path),
]
completed = None
max_krx_import_attempts = 4
for import_attempt in range(1, max_krx_import_attempts + 1):
    completed = subprocess.run(
        command, env=run_env, text=True, capture_output=True, timeout=3900
    )
    if completed.returncode == 0:
        break

    process_log = completed.stdout + "\n" + completed.stderr
    transient_krx_import_error = all(
        marker in process_log
        for marker in [
            "build_krx_session",
            "login_krx",
            "JSONDecodeError",
        ]
    )
    if (
        transient_krx_import_error
        and import_attempt < max_krx_import_attempts
    ):
        wait_seconds = 5 * import_attempt
        print(
            "KRX_IMPORT_RETRY "
            f"attempt={import_attempt}/{max_krx_import_attempts} "
            f"wait={wait_seconds}s"
        )
        time.sleep(wait_seconds)
        continue
    break

if completed.returncode != 0:
    print(completed.stdout[-6000:])
    print(completed.stderr[-6000:])
    raise RuntimeError(
        f"Notebook execution failed with exit code {completed.returncode}."
    )

executed = json.loads(executed_path.read_text(encoding="utf-8"))
errors = []
output_text = []
for cell_index, cell in enumerate(executed.get("cells", [])):
    for output in cell.get("outputs", []):
        if output.get("output_type") == "error":
            errors.append({
                "cell": cell_index,
                "ename": output.get("ename"),
                "evalue": output.get("evalue"),
            })
        text_value = output.get("text", "")
        if isinstance(text_value, list):
            text_value = "".join(text_value)
        if text_value:
            output_text.append(str(text_value))

if errors:
    raise RuntimeError(f"Executed notebook contains errors: {errors}")
combined_output = "\n".join(output_text)
for required_marker in [
    "AUTO_KRX_PREFLIGHT_OK",
    "MARKET_SMOKE_TEST_OK",
    "ACTUAL_HOLDINGS_PREVIEW_OK",
    "ORDER_REVIEW_PREVIEW_OK",
]:
    if required_marker not in combined_output:
        raise RuntimeError(
            f"Required success marker was not produced: {required_marker}"
        )

smoke_line = [
    line.strip() for line in combined_output.splitlines()
    if "MARKET_SMOKE_TEST_OK" in line
][-1]
preview_line = [
    line.strip() for line in combined_output.splitlines()
    if "ACTUAL_HOLDINGS_PREVIEW_OK" in line
][-1]
review_line = [
    line.strip() for line in combined_output.splitlines()
    if "ORDER_REVIEW_PREVIEW_OK" in line
][-1]

table_start = combined_output.find("ORDER_REVIEW_TABLE_BEGIN")
table_end = combined_output.find("ORDER_REVIEW_TABLE_END")
if table_start < 0 or table_end < table_start:
    raise RuntimeError("Detailed Order Review table was not produced.")
table_end += len("ORDER_REVIEW_TABLE_END")
order_table_block = combined_output[table_start:table_end]

print(smoke_line)
print(preview_line)
print(order_table_block)
print(review_line)
print("ORDER_REVIEW_EXECUTION_OK")
print(f"source_ref={SOURCE_REF}")
print(f"source_blob_sha={SOURCE_COMMIT_BLOB_SHA}")
print(f"executed_notebook={executed_path}")
print("run_daily_called=False")
print("persistent_writes=False")
